In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Generate synthetic wtg scada data

## Summary

Generate synthetic wind turbine scada data that exhibits more or less common data issues along with simple physical relations for demonstrational purpose.

## Results

### Key results

- Features:
  - Provides: Wind speed [m/s], active power [kW], rotor speed [min^-1], pitch angle [°].

### Details

- Physical relations:
  - Wind speed to power: 
    - According to a simple, piecwise linear power curve.
  - Wind speed to rotor speed:
    - Piecewise linear function such that:
      - Strictly below cut-in: zero.
      - Slightly below cut-in: Linear to interpolate between zero and following proportional relation.
      - Between cut-in and rated: Proportional w/ constant tip speed ratio.
      - Between rated and cut out: Constant at rated rotor speed.
      - Above cut-out: Rotor stopped.
    - Single discontinuity at cut-out.
  - Wind speed to pitch angle:
    - Below rated: Constant 1.
    - Between rated and cut-out: ~arccos((v_rated/v)**3). Constant chosen such that at 25m/s angle is approx 25° (constant w/o special interpretation).
    - Above cut-out: Fully pitched to avoid wind.

In [ ]:
import dataclasses
import numpy as np
import pandas as pd
import scipy.special
import scipy.interpolate
from numpy.typing import NDArray

In [ ]:
@dataclasses.dataclass
class WtgType:
    nominal_power: float
    rotor_diameter: float
    cut_in: float
    rated: float
    cut_out: float
    tsr: float


@dataclasses.dataclass
class Time:
    start: str
    freq: str | pd.Timedelta
    periods: int


DEFAULT_WTG = WtgType(nominal_power=5600, rotor_diameter=150, cut_in=3.0, cut_out=25.0, rated=13.0, tsr=8.0)
DEFAULT_TIME = Time(start="2026-02-14T04:00:00", freq="10min", periods=31)

In [ ]:
def compute_weibull_U_from_A(A: float, k: float) -> float:
    """Compute the mean of the Weibull distribution from its parameters.

    Parameters
    ----------
    A
        Scale parameter of the Weibull distribution.
    k
        Shape parameter of the Weibull distribution.

    Returns
    -------
    float
        Mean of the Weibull distribution.
    """
    return A * scipy.special.gamma(1 + 1 / k)


def compute_weibull_A_from_U(U: float, k: float) -> float:
    """Compute the scale parameter from mean and shape parameter of the Weibull distribution.

    Parameters
    ----------
    U
        Mean of the Weibull distribution.
    k
        Shape parameter of the Weibull distribution.

    Returns
    -------
    float
        Scale parameter of the Weibull distribution.
    """
    return U / scipy.special.gamma(1 + 1 / k)


def generate_random_weibull(size: tuple[int, ...], A: float, k: float, seed: int | None = None) -> NDArray[np.floating]:
    rng = np.random.default_rng(seed=seed)
    X = rng.random(size=size)
    wind_speeds = np.float_power(-np.log(X), 1 / k) * A
    return wind_speeds


def wind_speed_to_power(
    wtg_type: WtgType, wind_speeds: NDArray[np.floating] = np.linspace(0, 30, num=61)
) -> NDArray[np.floating]:
    t = np.array(
        [0, wtg_type.cut_in - 0.5, wtg_type.cut_in, wtg_type.rated, wtg_type.cut_out, wtg_type.cut_out + 0.5, 30]
    )
    p = np.array([0, 0, 0.04 * wtg_type.nominal_power, wtg_type.nominal_power, wtg_type.nominal_power, 0, 0])
    values = scipy.interpolate.make_interp_spline(t, p, k=1)(wind_speeds)
    return np.column_stack([wind_speeds, values])


def wind_speed_to_rotor_speed(wind_speeds, wtg_type: WtgType):
    rotor_speed = wtg_type.tsr / (np.pi * wtg_type.rotor_diameter) * wind_speeds
    rotor_speed_before_cut_in = wtg_type.tsr / (np.pi * wtg_type.rotor_diameter) * (wtg_type.cut_in - 0.5)
    rotor_speed_at_rated_wind_speed = wtg_type.tsr / (np.pi * wtg_type.rotor_diameter) * wtg_type.rated

    rotor_speed = np.where(
        wind_speeds < wtg_type.cut_in, (rotor_speed - rotor_speed_before_cut_in) / wtg_type.cut_in / 0.5, rotor_speed
    )
    rotor_speed = np.where(wind_speeds < wtg_type.cut_in - 0.5, 0, rotor_speed)

    rotor_speed = np.where(wind_speeds > wtg_type.rated, rotor_speed_at_rated_wind_speed, rotor_speed)
    rotor_speed = np.where(wind_speeds > wtg_type.cut_out, 0, rotor_speed)
    return rotor_speed * 60


def wind_speed_to_pitch_angle(wind_speeds, wtg_type: WtgType):
    rated_ratio = wtg_type.rated / wind_speeds
    pitch_angle = np.rad2deg(np.arccos(np.clip(rated_ratio**3, max=1))) * 0.3

    pitch_angle = np.where(wind_speeds < wtg_type.rated, 1, pitch_angle)
    pitch_angle = np.where(wind_speeds > wtg_type.cut_out, 90, pitch_angle)
    return pitch_angle


def generate_wtg_scada(A=10, k=2, time: Time = DEFAULT_TIME, wtg_type: WtgType = DEFAULT_WTG):
    index = pd.date_range(start=time.start, freq=time.freq, periods=time.periods)
    wind_speed = generate_random_weibull(size=len(index), A=A, k=k)
    df = pd.DataFrame(data={"wind_speed": wind_speed}, index=index)

    power = wind_speed_to_power(wind_speeds=df["wind_speed"].values, wtg_type=wtg_type)
    df["power"] = power[:, 1]

    df["rotor_speed"] = wind_speed_to_rotor_speed(df["wind_speed"], wtg_type=wtg_type)
    df["pitch_angle"] = wind_speed_to_pitch_angle(df["wind_speed"], wtg_type=wtg_type)
    return df

In [ ]:
def curtail_at_night(df: pd.Series, limit: float):
    df = df.copy()
    night_times = (df.index.hour >= 22) | (df.index.hour < 6)
    df = np.where(night_times, np.minimum(df.values, limit), df.values)
    return df


def curtail_to_zero(
    df: pd.DataFrame, count: int, k_max: int, columns_to_keep: list = ["wind_speed"], random_state=None
):
    df = df.copy()
    columns_to_zero = df.columns.difference(columns_to_keep)
    rng = np.random.default_rng(seed=random_state)

    for _ in range(count):
        length = rng.integers(3, k_max + 1)
        start = rng.integers(0, len(df) - length)
        end = start + length

        df.iloc[start:end, df.columns.get_indexer(columns_to_zero)] = 0
    return df


def freeze_periods(df: pd.DataFrame, count: int, k_max: int, random_state=None):
    df = df.copy()
    rng = np.random.default_rng(seed=random_state)
    for _ in range(count):
        length = rng.integers(3, k_max + 1)
        start = rng.integers(0, len(df) - length)
        end = start + length

        first_row = df.iloc[start, :]
        df.iloc[start:end] = first_row
    return df


def remove_rows(df: pd.DataFrame, count: int, k_max: int | None = None, p: float = 0.2, random_state=None):
    df = df.copy()
    rng = np.random.default_rng(seed=random_state)
    to_drop = np.zeros(len(df), bool)

    for _ in range(count):
        length = rng.geometric(p)
        length = length if k_max is None else max(length, k_max)
        start = rng.integers(0, len(df) - length)
        end = start + length

        to_drop[start:end] = True
    df_dropped_rows = df.loc[~to_drop]
    return df_dropped_rows


def mess_scada_data(df: pd.DataFrame):
    df = df.copy()
    df["power"] = curtail_at_night(df=df["power"], limit=3200)
    df = curtail_to_zero(df=df, count=3, k_max=6)
    # add self-consumption
    # insert flat periods
    df = insert_flat_periods(df=df, count=3, k_max=6)
    # add gaps
    df = remove_rows(df=df, count=3, k_max=4, p=0.2)
    # write random weird values
    return df

In [ ]:
A, k = 10, 2
print(compute_weibull_U_from_A(A=A, k=k))
Us = generate_random_weibull(size=(10,), A=A, k=k)
print(Us)
print(Us.mean())

In [ ]:
time = Time(start="2026-02-14T04:00:00", freq="10min", periods=151)
df = generate_wtg_scada(time=time)
mess_scada_data(df).loc[:, ["wind_speed", "rotor_speed", "pitch_angle"]].plot()